# 06 — Por qué son atractivos: las variables detrás del ranking

**Proyecto:** RADAR Cibest  
**Propósito:** mostrar los datos y scores dimensionales que explican las posiciones
del ranking. Un score sin contexto es un número; las dimensiones y variables son el argumento.

In [22]:
from __future__ import annotations
import sys, warnings, importlib, re
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

# ---------------------------------------------------------------------------
# Reload exhaustivo de todos los módulos del pipeline para evitar caché stale
# ---------------------------------------------------------------------------
import src.utils
import src.data_preparation.cleaning
import src.data_preparation.feature_engineering
import src.data_extraction.complementary
import src.scoring.weighting
import src.scoring.ranking
import src.scoring.gravity
import src.scoring.hybrid_scorer

for mod in [
    src.utils,
    src.data_preparation.cleaning,
    src.data_preparation.feature_engineering,
    src.data_extraction.complementary,
    src.scoring.weighting,
    src.scoring.ranking,
    src.scoring.gravity,
    src.scoring.hybrid_scorer,
]:
    importlib.reload(mod)

from src.utils import load_all_configs, resolve_data_path, get_variable_catalog
from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring

configs = load_all_configs()
variable_catalog = get_variable_catalog(configs["variables"])

In [23]:
# --- Paleta Cibest ---
C = {
    "black": "#1A1A1A", "white": "#FFFFFF", "gold": "#FDD923",
    "gold_light": "#FFF7D3", "gray": "#2C2A28", "gray_mid": "#666666",
    "gray_light": "#E5E5E5", "green": "#0BA682", "red": "#C62828", "amber": "#F39C12",
}

def insight_box(title, text):
    display(HTML(f'<div style="border-left:6px solid {C["gold"]}; background-color:{C["gold_light"]}; padding:14px 18px; margin:12px 0; font-family:Arial; color:{C["gray"]};"><b>{title}</b><br>{text}</div>'))

def style_table(df, gradient_cols=None, cmap="RdYlGn", fmt=None):
    s = df.style.set_table_styles([
        {"selector": "th", "props": [("background-color", C["gray"]), ("color", C["gold"]),
            ("font-weight", "bold"), ("text-align", "center"), ("padding", "8px"), ("font-family", "Arial")]},
        {"selector": "td", "props": [("padding", "6px 10px"), ("font-family", "Arial")]},
    ])
    if gradient_cols: s = s.background_gradient(subset=gradient_cols, cmap=cmap)
    if fmt: s = s.format(fmt)
    return s

In [24]:
# ---------------------------------------------------------------------------
# 1. Carga autocontenida: master → scoring completo → wide_raw + rankings
# ---------------------------------------------------------------------------
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

candidates = sorted(
    [path for path in raw_dir.glob("master_raw_*.parquet") if pattern.match(path.name)],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise FileNotFoundError("No se encontró master_raw_YYYYMMDD.parquet. Ejecute primero el notebook 01.")

master_path = candidates[0]
master = pd.read_parquet(master_path)

required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)
if missing_cols:
    raise ValueError(f"Master inválido. Faltan columnas requeridas: {sorted(missing_cols)}")

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

summary_master = pd.DataFrame({
    "métrica": ["archivo", "filas", "países", "variables", "fuentes", "año_min", "año_max", "incluye_gdp_growth"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        master["source"].nunique(),
        int(master["year"].min()),
        int(master["year"].max()),
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
    ],
})

display(style_table(summary_master))

results = run_full_scoring(master, configs, persist=False)

# wide_raw: valores absolutos por país (está en results, no en disco)
wide_raw = results["wide_raw"].copy()
if "country_iso3" not in wide_raw.columns:
    wide_raw = wide_raw.reset_index().rename(columns={"index": "country_iso3"})

# global_ranking: scores TOPSIS global con scores dimensionales
global_ranking = results["global_ranking"].copy()
if "country_iso3" not in global_ranking.columns:
    global_ranking = global_ranking.reset_index().rename(columns={"index": "country_iso3"})

# RADAR global
radar_global = results["radar_global"].copy()
if "country_iso3" not in radar_global.columns:
    radar_global = radar_global.reset_index().rename(columns={"index": "country_iso3"})
radar_global = radar_global.sort_values("radar_score", ascending=False)
if "rank" not in radar_global.columns:
    radar_global["rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

top_15 = radar_global.head(15)["country_iso3"].tolist()
top_10 = top_15[:10]
top_5 = top_15[:5]

print(f"Wide raw: {wide_raw.shape[0]} países × {wide_raw.shape[1]} columnas")
print(f"Variables disponibles: {[c for c in wide_raw.columns if c != 'country_iso3']}")
print(f"Top 5 RADAR: {', '.join(top_5)}")

,métrica,valor
0,archivo,master_raw_20260624.parquet
1,filas,1288
2,países,30
3,variables,45
4,fuentes,3
5,año_min,1996
6,año_max,2026
7,incluye_gdp_growth,Sí


2026-06-25 10:15:24.265 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-25 10:15:24.274 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 - Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-25 10:15:24.277 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 - Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-25 10:15:24.278 | INFO     | src.data_preparation.cleaning:appl

Wide raw: 24 países × 47 columnas
Variables disponibles: ['gdp_nominal', 'gdp_per_capita_ppp', 'gdp_growth', 'inflation_rate', 'population_total', 'urban_population_pct', 'unemployment_rate', 'current_account_gdp', 'public_debt_gdp', 'trade_openness', 'fdi_net_inflows_gdp', 'weighted_mean_applied_tariff_all_products', 'domestic_credit_private_gdp', 'account_ownership', 'interest_rate_spread', 'bank_npl_ratio', 'stock_market_cap_gdp', 'personal_remittances_gdp', 'bank_concentration_5', 'financial_system_deposits_gdp', 'bank_capital_rwa', 'regulatory_quality', 'government_effectiveness', 'rule_of_law', 'political_stability', 'voice_accountability', 'control_of_corruption', 'country_risk_premium', 'heritage_efi', 'internet_users_pct', 'mobile_subscriptions', 'digital_payment_adults_pct', 'secure_internet_servers_per_million', 'commercial_bank_branches_per_100k_adults', 'atms_per_100k_adults', 'ict_goods_exports_pct_total_goods_exports', 'geographic_distance_km', 'common_language_spanish',

In [25]:
# ---------------------------------------------------------------------------
# 2. Ejecutar scoring completo
# ---------------------------------------------------------------------------
print("\nEjecutando run_full_scoring()...")
results = run_full_scoring(master, configs, persist=False)
print("  ✓ Scoring completado")

2026-06-25 10:15:24.735 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)


2026-06-25 10:15:24.745 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 - Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-25 10:15:24.750 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 - Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-25 10:15:24.754 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:230 - Países con más datos stale: {'VEN': 12, 'BRB': 5, 'TTO': 4, 'CAN': 3, 'HTI': 3, 'BOL': 2, 'BLZ': 2, 'CHL': 2, 'CUB': 2, 'DOM': 2, 'ECU': 2, 'SUR':


Ejecutando run_full_scoring()...


2026-06-25 10:15:25.017 | INFO     | src.data_preparation.cleaning:impute_missing:335 - Imputacion (regional_median): 91 -> 0 celdas faltantes
2026-06-25 10:15:25.024 | INFO     | src.data_preparation.cleaning:run_cleaning:454 - Limpieza completada: 25 países x 45 variables | excluidos=['BRB', 'CUB', 'GUY', 'HTI', 'VEN']
2026-06-25 10:15:25.039 | INFO     | src.data_preparation.feature_engineering:apply_log_transform:73 - Log-transformacion (natural) aplicada a: ['gdp_nominal', 'population_total', 'geographic_distance_km', 'colombian_diaspora_stock', 'stock_market_cap_gdp', 'financial_system_deposits_gdp', 'domestic_credit_private_gdp', 'fdi_net_inflows_gdp', 'personal_remittances_gdp', 'secure_internet_servers_per_million', 'atms_per_100k_adults', 'commercial_bank_branches_per_100k_adults']
2026-06-25 10:15:25.050 | INFO     | src.data_preparation.feature_engineering:run_feature_engineering:136 - Feature engineering completado: 25 paises x 46 variables
2026-06-25 10:15:25.054 | INFO  

  ✓ Scoring completado


In [26]:
# ---------------------------------------------------------------------------
# 3. Extraer wide_raw, rankings y scores dimensionales
# ---------------------------------------------------------------------------

# --- wide_raw ---
wide_raw = results["wide_raw"].copy()

# Diagnóstico del índice
print(f"\nwide_raw original:")
print(f"  Shape: {wide_raw.shape}")
print(f"  Index name: {wide_raw.index.name}")
print(f"  Columnas ({len(wide_raw.columns)}): {list(wide_raw.columns[:10])}{'...' if len(wide_raw.columns) > 10 else ''}")

# Normalizar: country_iso3 siempre como columna
if wide_raw.index.name == "country_iso3":
    wide_raw = wide_raw.reset_index()
elif "country_iso3" not in wide_raw.columns and wide_raw.index.dtype == "object":
    wide_raw = wide_raw.reset_index().rename(columns={wide_raw.index.name or "index": "country_iso3"})

# --- global_ranking (scores dimensionales TOPSIS) ---
global_ranking = results["global_ranking"].copy()
if global_ranking.index.name == "country_iso3":
    global_ranking = global_ranking.reset_index()
elif "country_iso3" not in global_ranking.columns:
    global_ranking = global_ranking.reset_index().rename(columns={global_ranking.index.name or "index": "country_iso3"})

# --- RADAR global ---
radar_global = results["radar_global"].copy()
if radar_global.index.name == "country_iso3":
    radar_global = radar_global.reset_index()
elif "country_iso3" not in radar_global.columns:
    radar_global = radar_global.reset_index().rename(columns={radar_global.index.name or "index": "country_iso3"})
radar_global = radar_global.sort_values("radar_score", ascending=False)
if "rank" not in radar_global.columns:
    radar_global["rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

top_15 = radar_global.head(15)["country_iso3"].tolist()
top_10 = top_15[:10]
top_5 = top_15[:5]


wide_raw original:
  Shape: (24, 46)
  Index name: country_iso3
  Columnas (46): ['gdp_nominal', 'gdp_per_capita_ppp', 'gdp_growth', 'inflation_rate', 'population_total', 'urban_population_pct', 'unemployment_rate', 'current_account_gdp', 'public_debt_gdp', 'trade_openness']...


In [27]:
# ---------------------------------------------------------------------------
# 4. Inventario de variables disponibles en wide_raw
# ---------------------------------------------------------------------------
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{'='*60}")
print(f"INVENTARIO: {len(available_vars)} variables en wide_raw")
print(f"{'='*60}")

# Agrupar por dimensión del catálogo
dim_vars_available: Dict[str, List[str]] = {}
uncatalogued = []
for var in available_vars:
    meta = variable_catalog.get(var, None)
    if meta:
        dim = meta.get("dimension", "sin_dimension")
        dim_vars_available.setdefault(dim, []).append(var)
    else:
        uncatalogued.append(var)

for dim_name, dim_label in [("macro", "D1 Macro"), ("financial", "D2 Financiera"),
                             ("institutional", "D3 Institucional"), ("digital_tech", "D4 Digital-Tech"),
                             ("proximity", "D5 Proximidad")]:
    dim_v = dim_vars_available.get(dim_name, [])
    print(f"\n  {dim_label}: {len(dim_v)} variables")
    for v in dim_v:
        desc = variable_catalog[v].get("description", v)[:50]
        non_null = wide_raw[v].notna().sum()
        print(f"    {v}: {desc} ({non_null} países con dato)")

if uncatalogued:
    print(f"\n  Sin catálogo: {len(uncatalogued)} → {', '.join(uncatalogued[:10])}")

# Scores dimensionales disponibles
dim_score_cols = [c for c in ["score_macro", "score_financial", "score_institutional", "score_digital_tech"]
                  if c in global_ranking.columns]
print(f"\n  Scores dimensionales en global_ranking: {dim_score_cols}")
print(f"\nTop 5 RADAR: {', '.join(top_5)}")


INVENTARIO: 46 variables en wide_raw

  D1 Macro: 12 variables
    gdp_nominal: PIB nominal en USD corrientes (24 países con dato)
    gdp_per_capita_ppp: PIB per capita PPP en dolares internacionales corr (24 países con dato)
    gdp_growth: Crecimiento anual real del PIB (24 países con dato)
    inflation_rate: Inflacion anual de precios al consumidor (24 países con dato)
    population_total: Poblacion total (24 países con dato)
    urban_population_pct: Poblacion urbana como porcentaje del total (24 países con dato)
    unemployment_rate: Tasa de desempleo (%) (24 países con dato)
    current_account_gdp: Cuenta corriente como % del PIB (24 países con dato)
    public_debt_gdp: Deuda pública como % del PIB (24 países con dato)
    trade_openness: Apertura comercial (% del PIB) (24 países con dato)
    fdi_net_inflows_gdp: Inversión extranjera directa (% del PIB) (24 países con dato)
    weighted_mean_applied_tariff_all_products: Tarifa aplicada promedio ponderada para todos los  (

## 2. Radar chart: perfil dimensional de los top-5 (5 dimensiones incluyendo proximidad)

Cada país tiene un score TOPSIS parcial por dimensión estructural (macro, financiera,
institucional, digital-tecnológica) más el índice de proximidad con Colombia (IPC).
Este gráfico revela por qué cada país ocupa su posición en el ranking.

In [28]:
DIM_COLS = ["score_macro", "score_financial", "score_institutional", "score_digital_tech"]
DIM_LABELS_SHORT = ["Macro", "Financiera", "Institucional", "Digital-Tech"]
dim_available = [c for c in DIM_COLS if c in global_ranking.columns]

# Incorporar IPC como 5ª dimensión
ipc_df = results.get("ipc", None)
has_ipc = ipc_df is not None
if has_ipc:
    ipc_series = ipc_df.copy()
    if isinstance(ipc_series, pd.DataFrame):
        if "country_iso3" not in ipc_series.columns:
            ipc_series = ipc_series.reset_index().rename(columns={ipc_series.index.name or "index": "country_iso3"})
        ipc_col = [c for c in ipc_series.columns if c != "country_iso3"][0]
        ipc_map = ipc_series.set_index("country_iso3")[ipc_col].to_dict()
    elif isinstance(ipc_series, pd.Series):
        ipc_map = ipc_series.to_dict()
    else:
        ipc_map = {}
        has_ipc = False

if len(dim_available) >= 3:
    dim_data = global_ranking.set_index("country_iso3")[dim_available].copy()
    if has_ipc:
        dim_data["ipc"] = dim_data.index.map(ipc_map)
        all_dims = dim_available + ["ipc"]
        all_labels = DIM_LABELS_SHORT[:len(dim_available)] + ["Proximidad\nColombia"]
    else:
        all_dims = dim_available
        all_labels = DIM_LABELS_SHORT[:len(dim_available)]

    medians = dim_data[all_dims].median()

    # --- Colores Cibest diferenciados por país ---
    country_styles = {
        top_5[0]: {"color": C["gold"],    "width": 3.5, "dash": "solid",    "fill_opacity": 0.15},
        top_5[1]: {"color": C["green"],   "width": 3.0, "dash": "solid",    "fill_opacity": 0.12},
        top_5[2]: {"color": "#FFFFFF",    "width": 3.0, "dash": "dot",      "fill_opacity": 0.08},
        top_5[3]: {"color": C["amber"],   "width": 2.5, "dash": "dashdot",  "fill_opacity": 0.10},
        top_5[4]: {"color": "#4FC3F7",    "width": 2.5, "dash": "longdash", "fill_opacity": 0.08},
    }

    labels_closed = all_labels + [all_labels[0]]
    fig = go.Figure()

    for country in top_5:
        if country not in dim_data.index:
            continue
        style = country_styles.get(country, {"color": C["gray_mid"], "width": 2, "dash": "solid", "fill_opacity": 0.1})
        vals = dim_data.loc[country, all_dims].tolist()
        vals_closed = vals + [vals[0]]

        # Score RADAR para la leyenda
        r_score = radar_global.loc[radar_global["country_iso3"] == country, "radar_score"]
        r_label = f"{country} (RADAR {r_score.values[0]:.3f})" if len(r_score) > 0 else country

        fig.add_trace(go.Scatterpolar(
            r=vals_closed, theta=labels_closed, name=r_label,
            line=dict(color=style["color"], width=style["width"], dash=style["dash"]),
            fill="toself", fillcolor=f"rgba({int(style['color'].lstrip('#')[0:2], 16)},{int(style['color'].lstrip('#')[2:4], 16)},{int(style['color'].lstrip('#')[4:6], 16)},{style['fill_opacity']})",
            marker=dict(size=6, color=style["color"]),
        ))

    # Mediana del universo
    med_vals = medians.tolist() + [medians.tolist()[0]]
    fig.add_trace(go.Scatterpolar(
        r=med_vals, theta=labels_closed, name="Mediana universo",
        line=dict(color=C["gray_mid"], width=2, dash="dash"),
        marker=dict(size=0),
        fillcolor="rgba(0,0,0,0)",
    ))

    fig.update_layout(
        title=dict(
            text="Perfil dimensional de los top-5: no todos llegan al ranking por las mismas razones",
            font=dict(size=15),
        ),
        polar=dict(
            bgcolor=C["black"],
            radialaxis=dict(
                range=[0, 1], showticklabels=True, tickformat=".1f",
                gridcolor="rgba(255,255,255,0.15)", tickfont=dict(color=C["gray_mid"], size=10),
            ),
            angularaxis=dict(
                gridcolor="rgba(255,255,255,0.15)",
                tickfont=dict(color=C["white"], size=12, family="Arial"),
            ),
        ),
        paper_bgcolor=C["black"],
        font=dict(family="Arial", color=C["white"]),
        legend=dict(
            font=dict(size=12, color=C["white"]),
            bgcolor="rgba(0,0,0,0.5)",
            bordercolor=C["gray_mid"], borderwidth=1,
        ),
        height=620,
    )
    fig.show()

    insight_box("Lectura del radar",
        "PAN y CRI sobresalen en Proximidad pero quedan por debajo de la mediana en lo estructural. "
        "ESP, CHL mantienen perfil equilibrado en las 5 dimensiones. "
        "La asimetría entre proximidad y estructura es la tensión central del ranking RADAR.")

### Radar chart individual por país del top-5

Vista individual para presentaciones: cada país contra la mediana del universo.

In [29]:
if len(dim_available) >= 3:
    for country in top_5:
        if country not in dim_data.index:
            continue
        r_score = radar_global.loc[radar_global["country_iso3"] == country, "radar_score"]
        r_val = f"{r_score.values[0]:.3f}" if len(r_score) > 0 else "N/A"
        r_rank = radar_global.loc[radar_global["country_iso3"] == country, "rank"]
        r_pos = f"#{int(r_rank.values[0])}" if len(r_rank) > 0 else ""

        vals = dim_data.loc[country, all_dims].tolist() + [dim_data.loc[country, all_dims].tolist()[0]]

        fig = go.Figure()
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=labels_closed, name=country,
            line=dict(color=C["gold"], width=3),
            fill="toself", fillcolor=f"rgba(253,217,35,0.20)",
            marker=dict(size=7, color=C["gold"]),
        ))
        fig.add_trace(go.Scatterpolar(
            r=med_vals, theta=labels_closed, name="Mediana",
            line=dict(color=C["gray_mid"], width=2, dash="dash"),
            marker=dict(size=0),
        ))
        fig.update_layout(
            title=dict(text=f"{country} {r_pos} — Score RADAR: {r_val}", font=dict(size=14)),
            polar=dict(
                bgcolor=C["black"],
                radialaxis=dict(range=[0, 1], tickformat=".1f",
                                gridcolor="rgba(255,255,255,0.15)", tickfont=dict(color=C["gray_mid"])),
                angularaxis=dict(gridcolor="rgba(255,255,255,0.15)",
                                 tickfont=dict(color=C["white"], size=11)),
            ),
            paper_bgcolor=C["black"], font=dict(family="Arial", color=C["white"]),
            legend=dict(font=dict(color=C["white"]), bgcolor="rgba(0,0,0,0)"),
            height=420, showlegend=False,
        )
        fig.show()

In [30]:
# Tabla de scores dimensionales + IPC para top-15
if dim_available:
    dim_table = global_ranking[global_ranking["country_iso3"].isin(top_15)][
        ["country_iso3", "score", "rank"] + dim_available
    ].sort_values("rank").copy()
    if has_ipc:
        dim_table["ipc"] = dim_table["country_iso3"].map(ipc_map)
    dim_table = dim_table.rename(columns={
        "score": "TOPSIS_global", "score_macro": "Macro", "score_financial": "Financiera",
        "score_institutional": "Institucional", "score_digital_tech": "Digital-Tech",
        "ipc": "Proximidad",
    })
    grad_cols = [c for c in ["TOPSIS_global", "Macro", "Financiera", "Institucional", "Digital-Tech", "Proximidad"] if c in dim_table.columns]
    display(style_table(
        dim_table, gradient_cols=grad_cols,
        fmt={c: "{:.3f}" for c in dim_table.columns if dim_table[c].dtype in ["float64", "float32"]},
    ).set_caption("Scores dimensionales TOPSIS + Proximidad — Top 15 RADAR"))

,country_iso3,TOPSIS_global,rank,Macro,Financiera,Institucional,Digital-Tech,Proximidad
0,USA,0.703,1,0.691,0.648,0.786,0.716,0.286
1,CAN,0.693,2,0.685,0.573,0.905,0.634,0.174
2,ESP,0.644,3,0.604,0.604,0.725,0.712,0.000
3,CHL,0.609,4,0.519,0.554,0.763,0.645,0.264
4,URY,0.580,5,0.454,0.493,0.808,0.608,0.189
5,CRI,0.544,6,0.451,0.481,0.709,0.547,0.795
6,BHS,0.528,7,0.386,0.558,0.652,0.506,0.503
7,PAN,0.513,8,0.459,0.528,0.543,0.532,0.951
10,DOM,0.497,11,0.448,0.492,0.559,0.437,0.792
12,MEX,0.487,13,0.592,0.461,0.423,0.448,0.346


## 3. Heatmap dimensional: fortalezas y debilidades de un vistazo

In [31]:
if dim_available:
    heatmap_cols = dim_available.copy()
    heatmap_dim = global_ranking[global_ranking["country_iso3"].isin(top_15)].set_index("country_iso3")[heatmap_cols].copy()
    if has_ipc:
        heatmap_dim["ipc"] = heatmap_dim.index.map(ipc_map)
        heatmap_cols.append("ipc")
    heatmap_dim = heatmap_dim.rename(columns={
        "score_macro": "Macro", "score_financial": "Financiera",
        "score_institutional": "Institucional", "score_digital_tech": "Digital-Tech",
        "ipc": "Proximidad",
    })
    heatmap_dim = heatmap_dim.reindex(top_15)

    fig = px.imshow(
        heatmap_dim, color_continuous_scale="RdYlGn", aspect="auto",
        title="Heatmap dimensional: verde = fortaleza relativa, rojo = debilidad",
        text_auto=".2f",
    )
    fig.update_layout(height=550, xaxis_title="Dimensión", yaxis_title="País",
                      font=dict(family="Arial"))
    fig.show()

## 4. Variables absolutas disponibles en el modelo

Las secciones siguientes muestran las variables reales del wide_raw.
Solo se grafican las variables que existen en el dataset.

In [32]:
# Descubrir variables disponibles y mapearlas a dimensiones
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{len(available_vars)} variables disponibles en wide_raw:")

dim_vars_available: Dict[str, List[str]] = {}
for var in available_vars:
    meta = variable_catalog.get(var, {})
    dim = meta.get("dimension", "other")
    dim_vars_available.setdefault(dim, []).append(var)
    
for dim, vars_list in sorted(dim_vars_available.items()):
    print(f"  {dim}: {', '.join(vars_list)}")


46 variables disponibles en wide_raw:
  digital_tech: internet_users_pct, mobile_subscriptions, digital_payment_adults_pct, secure_internet_servers_per_million, commercial_bank_branches_per_100k_adults, atms_per_100k_adults, ict_goods_exports_pct_total_goods_exports
  financial: domestic_credit_private_gdp, account_ownership, interest_rate_spread, bank_npl_ratio, stock_market_cap_gdp, personal_remittances_gdp, bank_concentration_5, financial_system_deposits_gdp, bank_capital_rwa
  institutional: regulatory_quality, government_effectiveness, rule_of_law, political_stability, voice_accountability, control_of_corruption, country_risk_premium, heritage_efi
  macro: gdp_nominal, gdp_per_capita_ppp, gdp_growth, inflation_rate, population_total, urban_population_pct, unemployment_rate, current_account_gdp, public_debt_gdp, trade_openness, fdi_net_inflows_gdp, weighted_mean_applied_tariff_all_products
  other: cultural_distance_hofstede
  proximity: geographic_distance_km, common_language_spa

In [38]:
# Helper genérico: tabla + barplot horizontal de una variable
def barplot_variable(
    var_name: str,
    title: str,
    xlabel: str,
    reverse_color: bool = False,
    show_global_median: bool = True,
):
    if var_name not in wide_raw.columns:
        display(Markdown(f"*Variable `{var_name}` no disponible en wide_raw.*"))
        return

    plot_df = (
        wide_raw[wide_raw["country_iso3"].isin(top_15)][["country_iso3", var_name]]
        .dropna()
        .copy()
    )

    if plot_df.empty:
        display(Markdown(f"*No hay datos válidos para `{var_name}` en el Top 15.*"))
        return

    plot_df = plot_df.sort_values(var_name, ascending=False).reset_index(drop=True)

    # Agregar ranking RADAR para contexto
    plot_df = plot_df.merge(
        radar_global[["country_iso3", "radar_score", "rank"]]
        .rename(columns={"rank": "rank_RADAR"}),
        on="country_iso3",
        how="left",
    )

    plot_df["rank_variable"] = (
        plot_df[var_name]
        .rank(ascending=(not reverse_color), method="min")
        .astype(int)
    )

    # Tabla
    direction = variable_catalog.get(var_name, {}).get("direction", "positive")
    dir_label = "↓ menor = mejor" if direction == "negative" else "↑ mayor = mejor"

    display(Markdown(f"**{title}** ({dir_label})"))

    display(
        style_table(
            plot_df[["country_iso3", var_name, "rank_variable", "rank_RADAR"]]
            .rename(
                columns={
                    var_name: "Valor",
                    "rank_variable": "#Var",
                    "rank_RADAR": "#RADAR",
                }
            ),
            gradient_cols=["Valor"],
            cmap="RdYlGn_r" if reverse_color else "RdYlGn",
            fmt={
                "Valor": "{:,.2f}",
                "#Var": "{:.0f}",
                "#RADAR": "{:.0f}",
            },
        ).set_caption(f"{var_name}")
    )

    # Barplot
    plot_sorted = plot_df.sort_values(var_name, ascending=True)

    cscale = (
        [[0, C["green"]], [0.5, C["gold"]], [1, C["red"]]]
        if reverse_color
        else [[0, C["gray_light"]], [1, C["gold"]]]
    )

    fig = px.bar(
        plot_sorted,
        x=var_name,
        y="country_iso3",
        orientation="h",
        title=title,
        color=var_name,
        color_continuous_scale=cscale,
    )

    # Mediana del Top 15 mostrado
    median_top15 = plot_df[var_name].median()

    fig.add_vline(
        x=median_top15,
        line_width=2,
        line_dash="dash",
        line_color=C["gray"],
        annotation_text=f"Mediana Top 15: {median_top15:,.2f}",
        annotation_position="top right",
    )

    # Mediana del universo evaluado
    if show_global_median:
        median_all = pd.to_numeric(wide_raw[var_name], errors="coerce").median()

        fig.add_vline(
            x=median_all,
            line_width=2,
            line_dash="dot",
            line_color=C["red"],
            annotation_text=f"Mediana universo: {median_all:,.2f}",
            annotation_position="bottom right",
        )

    fig.update_layout(
        xaxis_title=xlabel,
        yaxis_title="",
        height=480,
        font=dict(family="Arial"),
    )

    fig.show()

## 5. Variables por dimensión

In [39]:
display(Markdown("### D1 — Macroeconómica"))
for var in dim_vars_available.get("macro", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D1 — Macroeconómica

**Cuenta corriente como % del PIB** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ECU,5.65,15,8
1,ESP,3.18,14,3
2,GTM,2.89,13,10
3,PER,2.21,12,11
4,PAN,1.93,11,1
5,CAN,-0.49,10,14
6,URY,-0.79,9,7
7,MEX,-0.90,8,9
8,CRI,-0.95,7,2
9,CHL,-1.47,6,5


**Inversión extranjera directa (% del PIB)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CRI,5.56,15,2
1,CHL,3.79,14,5
2,PAN,3.75,13,1
3,DOM,3.60,12,4
4,HND,3.53,11,13
5,CAN,2.81,10,14
6,SLV,2.61,9,12
7,ESP,2.48,8,3
8,MEX,2.45,7,9
9,PER,2.35,6,11


...

,country_iso3,Valor,#Var,#RADAR
0,ESP,10.38,1,3
1,BHS,9.21,2,15
2,CHL,8.97,3,5
3,PAN,8.36,4,1
4,URY,7.52,5,7
5,CAN,6.91,6,14
6,CRI,6.84,7,2
7,PER,5.12,8,11
8,DOM,5.09,9,4
9,HND,4.92,10,13


**Poblacion urbana como porcentaje del total** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,URY,95.60,15,7
1,CHL,89.00,14,5
2,PER,85.19,13,11
3,CAN,82.70,12,14
4,BHS,81.32,11,15
5,ESP,80.32,10,3
6,USA,80.12,9,6
7,MEX,79.75,8,9
8,CRI,79.31,7,2
9,SLV,75.07,6,12


**Tarifa aplicada promedio ponderada para todos los productos** (↓ menor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,BHS,16.34,1,15
1,PAN,5.93,2,1
2,ECU,5.19,3,8
3,URY,4.85,4,7
4,DOM,3.97,5,4
5,GTM,2.39,6,10
6,SLV,2.16,7,12
7,HND,2.11,8,13
8,MEX,1.62,9,9
9,USA,1.49,10,6


In [40]:
display(Markdown("### D2 — Industria Financiera"))
for var in dim_vars_available.get("financial", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D2 — Industria Financiera

**Adultos con cuenta en institucion financiera o servicio movil (%)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,98.40,14,14
1,ESP,98.38,13,3
2,USA,97.02,12,6
3,CHL,85.07,11,5
4,URY,73.75,10,7
5,CRI,71.35,9,2
6,DOM,64.78,8,4
7,ECU,64.51,7,8
8,PAN,64.10,6,1
9,PER,59.31,5,11


**Capital regulatorio bancario sobre activos ponderados por riesgo** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,DOM,17.73,14,4
1,MEX,17.70,13,9
2,URY,17.68,12,7
3,ESP,16.98,11,3
4,CRI,16.75,10,2
5,ECU,16.67,9,8
6,USA,16.28,8,6
7,GTM,16.24,7,10
8,CAN,16.10,6,14
9,PAN,15.71,5,1


...

,country_iso3,Valor,#Var,#RADAR
0,PER,7.78,1,11
1,GTM,7.46,2,10
2,HND,7.09,3,13
3,MEX,6.68,4,9
4,PAN,5.88,5,1
5,DOM,5.48,6,4
6,BHS,3.71,7,15
7,CRI,3.64,8,2
8,URY,2.61,9,7
9,CAN,2.60,10,14


**Remesas personales recibidas (% del PIB)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,HND,25.70,15,13
1,SLV,24.00,14,12
2,GTM,19.12,13,10
3,DOM,9.05,12,4
4,ECU,5.25,11,8
5,MEX,3.64,10,9
6,PER,1.71,9,11
7,CRI,0.76,8,2
8,PAN,0.61,7,1
9,BHS,0.42,6,15


**Capitalizacion bursatil domestica (% del PIB)** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,USA,216.29,10,6
1,CAN,150.40,9,14
2,CHL,79.33,8,5
3,ESP,43.69,7,3
4,MEX,32.07,6,9
5,PER,28.42,5,11
6,PAN,21.14,4,1
7,ECU,4.26,3,8
8,CRI,3.22,2,2
9,URY,1.38,1,7


In [41]:
display(Markdown("### D3 — Institucional"))
for var in dim_vars_available.get("institutional", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D3 — Institucional

**WGI: control de corrupcion** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.63,15,14
1,URY,1.53,14,7
2,BHS,1.23,13,15
3,CHL,1.11,12,5
4,USA,1.09,11,6
5,ESP,0.71,10,3
6,CRI,0.68,9,2
7,DOM,-0.27,8,4
8,PAN,-0.55,7,1
9,SLV,-0.58,6,12


**Prima de riesgo pais Damodaran / NYU Stern** (↓ menor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ECU,0.13,1,8
1,SLV,0.08,2,12
2,BHS,0.06,3,15
3,HND,0.06,3,13
4,CRI,0.04,5,2
5,DOM,0.04,5,4
6,GTM,0.03,7,10
7,PAN,0.03,8,1
8,MEX,0.02,9,9
9,PER,0.02,10,11


...

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.36,15,14
1,USA,1.35,14,6
2,CHL,0.78,13,5
3,URY,0.65,12,7
4,ESP,0.62,11,3
5,CRI,0.58,10,2
6,DOM,0.21,9,4
7,PER,0.08,8,11
8,PAN,-0.07,7,1
9,BHS,-0.11,6,15


**WGI: estado de derecho** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.46,15,14
1,USA,0.96,14,6
2,URY,0.95,13,7
3,ESP,0.93,12,3
4,CHL,0.69,11,5
5,CRI,0.62,10,2
6,BHS,0.25,9,15
7,DOM,-0.18,8,4
8,PAN,-0.26,7,1
9,PER,-0.51,6,11


**WGI: voz y rendicion de cuentas** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CAN,1.50,15,14
1,URY,1.23,14,7
2,ESP,1.15,13,3
3,CRI,1.01,12,2
4,CHL,0.99,11,5
5,USA,0.89,10,6
6,BHS,0.55,9,15
7,PAN,0.44,8,1
8,DOM,0.20,7,4
9,PER,0.05,6,11


In [42]:
display(Markdown("### D4 — Digital-Tecnológica"))
for var in dim_vars_available.get("digital_tech", []):
    label = variable_catalog.get(var, {}).get("description", var)
    direction = variable_catalog.get(var, {}).get("direction", "positive")
    barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                     reverse_color=(direction == "negative"))

### D4 — Digital-Tecnológica

**Cajeros automáticos por cada 100.000 adultos** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,URY,314.77,15,7
1,CAN,188.53,14,14
2,USA,173.19,13,6
3,PER,128.12,12,11
4,BHS,116.55,11,15
5,ESP,87.12,10,3
6,PAN,72.60,9,1
7,MEX,68.68,8,9
8,CRI,63.05,7,2
9,CHL,47.62,6,5


**Sucursales de bancos comerciales por cada 100.000 adultos** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,ESP,32.43,15,3
1,USA,26.63,14,6
2,GTM,22.83,13,10
3,BHS,21.70,12,15
4,CAN,19.02,11,14
5,PAN,18.59,10,1
6,CRI,15.84,9,2
7,HND,15.58,8,13
8,MEX,12.18,7,9
9,DOM,10.84,6,4


...

,country_iso3,Valor,#Var,#RADAR
0,ESP,95.76,15,3
1,CHL,95.59,14,5
2,USA,94.69,13,6
3,CAN,94.35,12,14
4,BHS,92.47,11,15
5,URY,91.99,10,7
6,DOM,91.00,9,4
7,CRI,87.17,8,2
8,MEX,83.12,7,9
9,PER,81.96,6,11


**Suscripciones moviles por 100 habitantes** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,SLV,176.52,15,12
1,PAN,156.59,14,1
2,URY,145.50,13,7
3,CRI,136.02,12,2
4,CHL,132.66,11,5
5,ESP,130.26,10,3
6,PER,124.62,9,11
7,MEX,116.49,8,9
8,USA,113.19,7,6
9,GTM,112.52,6,10


**Servidores seguros de internet por millon de personas** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,USA,"196,554.13",15,6
1,CAN,"39,712.75",14,14
2,ESP,"29,986.38",13,3
3,CHL,"11,912.61",12,5
4,BHS,"6,354.62",11,15
5,URY,"4,640.66",10,7
6,PAN,"2,234.49",9,1
7,CRI,"1,999.84",8,2
8,PER,826.79,7,11
9,MEX,531.66,6,9


In [43]:
display(Markdown("### D5 — Proximidad y otras"))
for dim_key in sorted(dim_vars_available.keys()):
    if dim_key in ("macro", "financial", "institutional", "digital_tech"):
        continue
    for var in dim_vars_available[dim_key]:
        label = variable_catalog.get(var, {}).get("description", var)
        direction = variable_catalog.get(var, {}).get("direction", "positive")
        barplot_variable(var, f"{label}", var.replace("_", " ").title(),
                         reverse_color=(direction == "negative"))

### D5 — Proximidad y otras

**Salidas de colombianos hacia el pais destino** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,USA,"256,327.00",15,6
1,ESP,"120,344.00",14,3
2,PAN,"56,596.00",13,1
3,MEX,"45,028.00",12,9
4,ECU,"43,682.00",11,8
5,CHL,"39,385.00",10,5
6,DOM,"30,352.00",9,4
7,CAN,"20,715.00",8,14
8,PER,"17,820.00",7,11
9,CRI,"9,341.00",6,2


**Espanol como idioma oficial o ampliamente compartido** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,CHL,1.00,4,5
1,CRI,1.00,4,2
2,DOM,1.00,4,4
3,ECU,1.00,4,8
4,ESP,1.00,4,3
5,GTM,1.00,4,10
6,HND,1.00,4,13
7,MEX,1.00,4,9
8,PAN,1.00,4,1
9,PER,1.00,4,11


...

,country_iso3,Valor,#Var,#RADAR
0,MEX,69.00,14,9
1,DOM,65.00,13,4
2,ECU,63.00,12,8
3,USA,62.00,11,6
4,CAN,52.00,10,14
5,PAN,44.00,9,1
6,ESP,42.00,7,3
7,PER,42.00,7,11
8,HND,40.00,5,13
9,SLV,40.00,5,12


**Hofstede: power distance index** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,GTM,95.00,13,10
1,PAN,95.00,13,1
2,MEX,81.00,12,9
3,HND,80.00,11,13
4,ECU,78.00,10,8
5,SLV,66.00,9,12
6,DOM,65.00,8,4
7,PER,64.00,7,11
8,CHL,63.00,6,5
9,URY,61.00,5,7


**Hofstede: uncertainty avoidance** (↑ mayor = mejor)

,country_iso3,Valor,#Var,#RADAR
0,GTM,98.00,13,10
1,URY,98.00,13,7
2,SLV,94.00,12,12
3,PER,87.00,11,11
4,CHL,86.00,7,5
5,CRI,86.00,7,2
6,ESP,86.00,7,3
7,PAN,86.00,7,1
8,MEX,82.00,6,9
9,ECU,67.00,5,8


## 6. Fichas top-5: dimensión más fuerte y más débil

In [44]:
if dim_available:
    for country in top_5:
        if country not in global_ranking["country_iso3"].values:
            continue
        row = global_ranking[global_ranking["country_iso3"] == country].iloc[0]
        rank_pos = int(row.get("rank", 0))
        radar_score = radar_global[radar_global["country_iso3"] == country]["radar_score"].values
        radar_val = f"{radar_score[0]:.3f}" if len(radar_score) > 0 else "N/A"

        dim_scores = {DIM_LABELS_SHORT[DIM_COLS.index(c)]: row[c] for c in dim_available if c in row.index}
        strongest = max(dim_scores, key=dim_scores.get)
        weakest = min(dim_scores, key=dim_scores.get)

        display(Markdown(f"### {country} — #{rank_pos} TOPSIS | RADAR: {radar_val}"))
        ficha = pd.DataFrame([{
            "Dimensión": k, "Score": f"{v:.3f}",
            "Rol": "⬆ Fortaleza" if k == strongest else ("⬇ Debilidad" if k == weakest else "—"),
        } for k, v in dim_scores.items()])
        display(style_table(ficha))

NameError: name 'DIM_LABELS_SHORT' is not defined

## 7. Scatter estratégico: score institucional vs score financiero

Los ejes representan las dos dimensiones con mayor peso en el modelo (30% cada una).
El tamaño del punto es proporcional al score RADAR compuesto.

In [ ]:
if "score_institutional" in global_ranking.columns and "score_financial" in global_ranking.columns:
    scatter_df = global_ranking[global_ranking["country_iso3"].isin(top_15)].merge(
        radar_global[["country_iso3", "radar_score"]], on="country_iso3", how="left"
    )
    fig = px.scatter(
        scatter_df, x="score_institutional", y="score_financial",
        text="country_iso3", size="radar_score", color="radar_score",
        color_continuous_scale=[[0, C["gray_light"]], [0.5, C["gold"]], [1, C["green"]]],
        title="Institucionalidad vs profundidad financiera: los dos pilares del atractivo estructural",
    )
    fig.update_traces(textposition="top center", textfont_size=10)

    # Líneas de cuadrante en 0.5
    fig.add_hline(y=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)
    fig.add_vline(x=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)

    # Etiquetas de cuadrante
    fig.add_annotation(x=0.85, y=0.92, text="Mercado maduro",
                       showarrow=False, font=dict(size=11, color=C["green"]), opacity=0.7)
    fig.add_annotation(x=0.15, y=0.92, text="Financiero pero\ninstitución débil",
                       showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
    fig.add_annotation(x=0.85, y=0.08, text="Institucional pero\nfinancieramente limitado",
                       showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
    fig.add_annotation(x=0.15, y=0.08, text="Corredor / Emergente",
                       showarrow=False, font=dict(size=11, color=C["red"]), opacity=0.7)

    fig.update_layout(
        xaxis_title="Score institucional (WGI + Prima de Riesgo Pais + Libertad económica)",
        yaxis_title="Score industria financiera",
        xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
        height=580, font=dict(family="Arial"),
    )
    fig.show()

    insight_box("Cuatro cuadrantes",
        "Arriba-derecha: mercados maduros con institucionalidad y profundidad (USA, CAN, CHL, ESP). "
        "Abajo-izquierda: mercados de corredor donde la oportunidad viene de proximidad. "
        "Los cuadrantes mixtos señalan mercados con una fortaleza aprovechable pero una restricción crítica.")

## 8. Scatter integral: entorno completo vs profundidad financiera

El eje X combina las cuatro dimensiones no financieras (macro, institucional,
digital-tecnológica y proximidad) en un promedio simple. El eje Y es la
dimensión financiera. Esto permite ver el comportamiento de los cinco
componentes del radar en un solo plano.

In [ ]:
env_dims = [c for c in ["score_macro", "score_institutional", "score_digital_tech"] if c in global_ranking.columns]
if len(env_dims) >= 2 and "score_financial" in global_ranking.columns:
   scatter2 = global_ranking[global_ranking["country_iso3"].isin(top_15)].copy()
   # Promedio de dimensiones estructurales no financieras
   scatter2["entorno_estructural"] = scatter2[env_dims].mean(axis=1)
   # Agregar IPC si disponible
   if has_ipc:
       scatter2["ipc_val"] = scatter2["country_iso3"].map(ipc_map)
       # Promedio ponderado: 75% dimensiones TOPSIS + 25% proximidad (reparto 5% de la tendencia (10%) a cada factor
       scatter2["entorno_integral"] = 0.75 * scatter2["entorno_estructural"] + 0.25 * scatter2["ipc_val"].fillna(0)
       x_col = "entorno_integral"
       x_label = "Entorno integral (Macro + Institucional + Digital + Proximidad)"
   else:
       x_col = "entorno_estructural"
       x_label = "Entorno estructural (Macro + Institucional + Digital)"
   scatter2 = scatter2.merge(
       radar_global[["country_iso3", "radar_score"]], on="country_iso3", how="left"
   )
   fig = px.scatter(
       scatter2, x=x_col, y="score_financial",
       text="country_iso3", size="radar_score", color="radar_score",
       color_continuous_scale=[[0, C["gray_light"]], [0.5, C["gold"]], [1, C["green"]]],
       title="Entorno integral vs profundidad financiera: la vista completa del radar en un plano",
   )
   fig.update_traces(textposition="top center", textfont_size=10)
   # Líneas de cuadrante en 0.5
   fig.add_hline(y=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)
   fig.add_vline(x=0.5, line_dash="dash", line_color=C["gray_mid"], line_width=1.5, opacity=0.6)
   # Etiquetas de cuadrante
   fig.add_annotation(x=0.85, y=0.92, text="Atractivo integral",
                      showarrow=False, font=dict(size=11, color=C["green"]), opacity=0.7)
   fig.add_annotation(x=0.15, y=0.92, text="Financiero pero\nentorno débil",
                      showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
   fig.add_annotation(x=0.85, y=0.08, text="Buen entorno pero\nsin profundidad financiera",
                      showarrow=False, font=dict(size=10, color=C["amber"]), opacity=0.7)
   fig.add_annotation(x=0.15, y=0.08, text="Baja atractividad\nestructural",
                      showarrow=False, font=dict(size=11, color=C["red"]), opacity=0.7)
   fig.update_layout(
       xaxis_title=x_label,
       yaxis_title="Score industria financiera",
       xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
       height=580, font=dict(family="Arial"),
   )
   fig.show()
   # Tabla de datos del scatter
   scatter2_display = scatter2[["country_iso3", x_col, "score_financial", "radar_score"]].sort_values("radar_score", ascending=False)
   scatter2_display = scatter2_display.rename(columns={
       x_col: "Entorno", "score_financial": "Financiera", "radar_score": "RADAR",
   })
   display(style_table(
       scatter2_display,
       gradient_cols=["Entorno", "Financiera", "RADAR"], cmap="RdYlGn",
       fmt={"Entorno": "{:.3f}", "Financiera": "{:.3f}", "RADAR": "{:.3f}"},
   ).set_caption("Datos del scatter integral"))
   insight_box("Lectura integral",
       "Este gráfico comprime las 5 dimensiones del radar en 2 ejes. "
       "Los países arriba-derecha combinan entorno favorable con sistema financiero profundo. "
       "Los países que suben por proximidad (PAN, CRI) se desplazan a la derecha respecto al scatter anterior. "
       "La diferencia entre ambos scatters revela cuánto aporta la proximidad a cada país.")

,country_iso3,Entorno,Financiera,RADAR
7,PAN,0.621,0.528,0.691
5,CRI,0.625,0.481,0.653
2,ESP,0.510,0.604,0.627
8,DOM,0.559,0.492,0.620
3,CHL,0.548,0.554,0.569
0,USA,0.620,0.648,0.552
4,URY,0.515,0.493,0.539
13,ECU,0.509,0.510,0.538
9,MEX,0.452,0.461,0.533
12,GTM,0.407,0.469,0.530


In [ ]:
# Descubrir variables disponibles y mapearlas a dimensiones
available_vars = [c for c in wide_raw.columns if c != "country_iso3"]
print(f"\n{len(available_vars)} variables disponibles en wide_raw:")

dim_vars_available: Dict[str, List[str]] = {}
for var in available_vars:
    meta = variable_catalog.get(var, {})
    dim = meta.get("dimension", "other")
    dim_vars_available.setdefault(dim, []).append(var)
    
for dim, vars_list in sorted(dim_vars_available.items()):
    print(f"  {dim}: {', '.join(vars_list)}")


12 variables disponibles en wide_raw:
  institutional: country_risk_premium, heritage_efi
  other: cultural_distance_hofstede
  proximity: geographic_distance_km, common_language_spanish, hofstede_pdi, hofstede_idv, hofstede_mas, hofstede_uai, hofstede_lto, hofstede_ivr, colombian_diaspora_stock


## Síntesis Ejecutiva

Los scores dimensionales revelan que no todos los países llegan al ranking por las
mismas razones. Los mercados maduros (USA, CAN, CHL, ESP) dominan en institucionalidad
y profundidad financiera. Los mercados de corredor (PAN, CRI, DOM, ECU) compensan
debilidad estructural con proximidad. El radar chart hace visible esta distinción
con más claridad que un ranking plano.